# ChainScore — Stress Testing & What-If Scenarios

This notebook evaluates how credit scores respond to macroeconomic shocks and behavioral changes.
It applies synthetic perturbations to the feature matrix and re-scores without fetching new on-chain data.

**Scenarios:**
1. ETH market crash (-40% collateral value)
2. Mass Aave withdrawal (protocol exposure reset)
3. Activity freeze (dormancy — wallet stops transacting)
4. Historical crisis replay: LUNA collapse (May 2022), FTX (Nov 2022), USDC depeg (Mar 2023)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white', 'font.family': 'monospace'})

In [ ]:
# Load models and feature matrix
lr  = joblib.load('../models/logistic_regression.pkl')
lgb = joblib.load('../models/lightgbm.pkl')

feature_cols = pd.read_json('../models/feature_columns.json', typ='series').tolist()
df = pd.read_parquet('../data/processed/feature_matrix.parquet')

X = df[feature_cols].copy()
y = df['label'].values

print(f'Dataset: {len(df):,} wallets | {y.mean():.1%} default rate')
print(f'Features: {len(feature_cols)}')

In [ ]:
def pd_to_score(pd_val: float) -> int:
    """Convert probability of default to 0-1000 ChainScore."""
    return int(round((1 - pd_val) * 1000))

def score_scenario(X_scenario: pd.DataFrame, label: str) -> pd.Series:
    pd_lr  = lr.predict_proba(X_scenario)[:, 1]
    scores = np.array([pd_to_score(p) for p in pd_lr])
    return pd.Series(scores, name=label)

# Baseline
baseline_scores = score_scenario(X, 'Baseline')
print(f'Baseline — mean score: {baseline_scores.mean():.0f} | median: {baseline_scores.median():.0f}')

## Scenario 1 — ETH Market Crash (-40%)

Simulates a sharp ETH price decline. Collateral value drops, reducing net flows and max transaction values.
Mimics conditions observed during LUNA collapse (May 2022) and FTX (Nov 2022).

In [ ]:
CRASH_FACTOR = 0.60  # -40% ETH price

X_crash = X.copy()
for col in ['eth_sent_total', 'eth_received_total', 'net_eth_flow',
            'avg_tx_value_eth', 'max_tx_value_eth', 'std_tx_value_eth']:
    if col in X_crash.columns:
        X_crash[col] *= CRASH_FACTOR

crash_scores = score_scenario(X_crash, 'ETH Crash -40%')
delta = crash_scores - baseline_scores

print(f'ETH Crash -40%')
print(f'  Mean score change:   {delta.mean():+.1f} pts')
print(f'  Wallets downgraded:  {(delta < -50).sum():,} ({(delta < -50).mean():.1%})')
print(f'  New high-risk share: {(crash_scores < 500).mean():.1%} (was {(baseline_scores < 500).mean():.1%})')

## Scenario 2 — Mass Aave Withdrawal (Protocol Exposure Reset)

Simulates wallets exiting Aave entirely — borrow/repay/deposit counts zeroed out.
Equivalent to a protocol liquidity crisis forcing users to withdraw.

In [ ]:
X_aave_exit = X.copy()
aave_cols = ['is_aave_user', 'aave_interaction_count', 'aave_deposit_count',
             'aave_borrow_count', 'aave_repay_count', 'aave_withdraw_count', 'repay_to_borrow_ratio']
for col in aave_cols:
    if col in X_aave_exit.columns:
        X_aave_exit[col] = 0

aave_scores = score_scenario(X_aave_exit, 'Aave Exit')
delta_aave = aave_scores - baseline_scores

# Only show impact on Aave users
was_aave = X['is_aave_user'] == 1
print(f'Aave Exit — impact on {was_aave.sum():,} Aave users')
print(f'  Mean score change:   {delta_aave[was_aave].mean():+.1f} pts')
print(f'  Median score change: {delta_aave[was_aave].median():+.1f} pts')
print(f'  Wallets downgraded > 100pts: {(delta_aave[was_aave] < -100).sum():,}')

## Scenario 3 — Activity Freeze (90-day Dormancy)

Simulates wallets going silent — recent_tx_ratio drops to 0, dormancy periods increase.
Relevant for identifying counterparties that have stopped managing their positions.

In [ ]:
X_dormant = X.copy()
if 'recent_tx_ratio' in X_dormant.columns:
    X_dormant['recent_tx_ratio'] = 0.0
if 'dormancy_periods_30d' in X_dormant.columns:
    X_dormant['dormancy_periods_30d'] = X_dormant['dormancy_periods_30d'] + 3
if 'activity_regularity' in X_dormant.columns:
    X_dormant['activity_regularity'] *= 0.3

dormant_scores = score_scenario(X_dormant, 'Dormancy')
delta_dormant = dormant_scores - baseline_scores

print(f'Activity Freeze (90-day dormancy)')
print(f'  Mean score change:   {delta_dormant.mean():+.1f} pts')
print(f'  Wallets downgraded:  {(delta_dormant < 0).sum():,} ({(delta_dormant < 0).mean():.1%})')

## Combined shock — Portfolio VaR under stress

Combines ETH crash + Aave exit: worst-case scenario for a DeFi-heavy portfolio.

In [ ]:
X_combined = X_crash.copy()
for col in aave_cols:
    if col in X_combined.columns:
        X_combined[col] = 0

combined_scores = score_scenario(X_combined, 'Combined Shock')
delta_combined = combined_scores - baseline_scores

print('Combined Shock (ETH -40% + Aave exit)')
print(f'  Mean score change:    {delta_combined.mean():+.1f} pts')
print(f'  VaR 95% (score):      {np.percentile(combined_scores, 5):.0f} (was {np.percentile(baseline_scores, 5):.0f})')
print(f'  High-risk share:      {(combined_scores < 500).mean():.1%} (was {(baseline_scores < 500).mean():.1%})')
print(f'  Very-high-risk share: {(combined_scores < 300).mean():.1%} (was {(baseline_scores < 300).mean():.1%})')

## Visualization — Score distribution shift under stress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: distribution comparison
ax = axes[0]
bins = np.linspace(0, 1000, 41)
for scores, label, color, alpha in [
    (baseline_scores, 'Baseline', '#2563eb', 0.7),
    (crash_scores,    'ETH -40%', '#f59e0b', 0.6),
    (combined_scores, 'Combined shock', '#dc2626', 0.5),
]:
    ax.hist(scores, bins=bins, alpha=alpha, label=label, color=color, density=True)

for x, label in [(300, 'Very High'), (500, 'High'), (650, 'Medium')]:
    ax.axvline(x, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.text(x + 5, ax.get_ylim()[1] * 0.95 if ax.get_ylim()[1] > 0 else 0.003,
            label, fontsize=7, color='gray', va='top')

ax.set_xlabel('ChainScore')
ax.set_ylabel('Density')
ax.set_title('Score Distribution Under Stress')
ax.legend(fontsize=9)

# Right: mean score change per scenario
ax2 = axes[1]
scenarios = ['ETH\nCrash -40%', 'Aave\nExit', 'Activity\nFreeze', 'Combined\nShock']
deltas = [
    delta.mean(),
    delta_aave.mean(),
    delta_dormant.mean(),
    delta_combined.mean(),
]
colors = ['#f59e0b', '#8b5cf6', '#06b6d4', '#dc2626']
bars = ax2.barh(scenarios, deltas, color=colors, alpha=0.8)
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_xlabel('Mean Score Change (pts)')
ax2.set_title('Impact per Scenario')
for bar, d in zip(bars, deltas):
    ax2.text(d - 1 if d < 0 else d + 1, bar.get_y() + bar.get_height()/2,
             f'{d:+.1f}', va='center', ha='right' if d < 0 else 'left', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/stress_testing.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/stress_testing.png')

## Historical Crisis Replay

We proxy three real crises using the perturbation patterns that best match their observed on-chain effects:

| Crisis | Date | ETH drawdown | Aave TVL drop | Proxy scenario |
|---|---|---|---|---|
| LUNA collapse | May 2022 | -55% | -60% | ETH -55% + partial Aave exit |
| FTX collapse | Nov 2022 | -25% | -20% | ETH -25% + activity freeze |
| USDC depeg | Mar 2023 | -8% | -15% | Mild combined shock |

In [ ]:
crises = {
    'LUNA (May 2022)': {'eth_factor': 0.45, 'aave_factor': 0.40, 'dormancy_add': 1},
    'FTX (Nov 2022)':  {'eth_factor': 0.75, 'aave_factor': 0.80, 'dormancy_add': 2},
    'USDC (Mar 2023)': {'eth_factor': 0.92, 'aave_factor': 0.85, 'dormancy_add': 0},
}

eth_value_cols = ['eth_sent_total', 'eth_received_total', 'net_eth_flow',
                  'avg_tx_value_eth', 'max_tx_value_eth', 'std_tx_value_eth']
aave_count_cols = ['aave_deposit_count', 'aave_borrow_count', 'aave_repay_count',
                   'aave_withdraw_count', 'aave_interaction_count']

print(f'{"Crisis":<20} {"Mean Δscore":>12} {"High-risk %":>12} {"Very-high %":>12}')
print('-' * 60)

crisis_results = {}
for name, params in crises.items():
    Xc = X.copy()
    for col in eth_value_cols:
        if col in Xc.columns:
            Xc[col] *= params['eth_factor']
    for col in aave_count_cols:
        if col in Xc.columns:
            Xc[col] *= params['aave_factor']
    if 'dormancy_periods_30d' in Xc.columns:
        Xc['dormancy_periods_30d'] += params['dormancy_add']
    
    s = score_scenario(Xc, name)
    crisis_results[name] = s
    d = s - baseline_scores
    print(f'{name:<20} {d.mean():>+11.1f} {(s < 500).mean():>11.1%} {(s < 300).mean():>11.1%}')

print(f'{"Baseline":<20} {0:>+11.1f} {(baseline_scores < 500).mean():>11.1%} {(baseline_scores < 300).mean():>11.1%}')